# Construccion de modelos de Machine Learning

Entrena los tres modelos de clasificacion que pide el enunciado sobre la matriz
de predictores del ejercicio 3: Regresion Logistica, Random Forest y Gradient
Boosting.

La logica vive en `src/modelos.py`; aqui se invoca, se muestra el resultado y se
explican las decisiones. La evaluacion comparativa esta en el cuaderno siguiente.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Abra Jupyter desde la raiz de Laboratorio 4')

sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Image, display

from src import modelos as md
from src.features import leer_features, columnas_predictoras

print(f'Semilla del laboratorio:   {md.SEMILLA}')
print(f'Fraccion de prueba:        {md.FRACCION_PRUEBA:.0%}')
print(f'Beta del F-score de ajuste:{md.BETA_F:.0f}')

Semilla del laboratorio:   42
Fraccion de prueba:        30%
Beta del F-score de ajuste:2


## 1. Punto de partida

La matriz de predictores viene del ejercicio 3 y ya trae la variable respuesta
`cyano_alta`. No se toca aqui: si algo esta mal hay que corregirlo alla y
volver a construirla, no parchearlo en este cuaderno.

In [2]:
matriz = leer_features()
predictores = columnas_predictoras(matriz)

print(f'Observaciones: {len(matriz):,}')
print(f'Predictores:   {len(predictores)}')
print()
print('Distribucion de la variable respuesta')
conteo = matriz['cyano_alta'].value_counts().sort_index()
display(pd.DataFrame({
    'n': conteo,
    'porcentaje': (100 * conteo / len(matriz)).round(4),
}))

Observaciones: 492,663
Predictores:   17

Distribucion de la variable respuesta


,n,porcentaje
cyano_alta,,
0,486298,98.708
1,6365,1.292


El desbalance es severo: poco mas del 1 por ciento de las celdas supera el
umbral de 10 microgramos por litro. Eso condiciona tres decisiones que se toman
a continuacion, y conviene dejarlas explicitas antes de entrenar nada.

**La division debe ser estratificada.** Con 1.29 por ciento de positivos, una
division aleatoria simple podria dejar el conjunto de prueba con muy pocos casos
de alta presencia y volver inestable cualquier metrica que se calcule sobre el.

**Los tres modelos deben compensar el desbalance.** Sin reponderar la clase
positiva, la forma mas facil de minimizar el error es predecir siempre ausencia,
que ya acierta el 98.7 por ciento de las veces.

**El criterio de ajuste no puede ser Accuracy.** Se usa F-beta con beta = 2, que
pesa el Recall cuatro veces mas que la Precision. La justificacion ambiental
completa esta en el cuaderno de evaluacion.

## 2. Division 70 / 30 y su persistencia

El enunciado pide entrenar con el 70 por ciento y probar con el 30, y ademas
mantener el mismo conjunto de prueba para que la comparacion entre modelos sea
justa. Eso ultimo no es solo una recomendacion de este cuaderno: los ejercicios
de validacion espacial, generalizacion entre lagos e interpretabilidad tienen
que evaluar sobre exactamente el mismo conjunto.

Por eso la particion se persiste en `data/processed/ml/particion_70_30.parquet`
con `indice`, `lago`, `fecha` y `particion`. Las columnas `lago` y `fecha` se
arrastran porque la validacion espacial y la temporal necesitan agrupar por
ellas y la matriz de predictores no las conserva.

In [3]:
particion = md.construir_particion(matriz)
md.escribir_particion(particion)

resumen = particion.groupby('particion').size().rename('observaciones').to_frame()
resumen['porcentaje'] = (100 * resumen['observaciones'] / len(particion)).round(2)
display(resumen)

datos = md.dividir(matriz, particion)
for etiqueta in ('entrenamiento', 'prueba'):
    y = datos[f'y_{etiqueta}']
    print(f'{etiqueta:14s} {len(y):>8,} observaciones  {int(y.sum()):>6,} positivas  {100*y.mean():.4f} %')

,observaciones,porcentaje
particion,,
entrenamiento,344864,70.0
prueba,147799,30.0


entrenamiento   344,864 observaciones   4,455 positivas  1.2918 %
prueba          147,799 observaciones   1,910 positivas  1.2923 %


Las dos tasas de positivos coinciden hasta el cuarto decimal, que es lo que se
espera de una division estratificada y lo que comprueba `modelos.py verificar`.

## 3. Los tres modelos y su espacio de busqueda

**Regresion Logistica.** Va dentro de un `Pipeline` con `StandardScaler` porque
las variables estan en unidades muy distintas, de reflectancia entre 0 y 1 a
metros en decenas de miles. El escalado dentro del pipeline garantiza que se
ajuste solo con el pliegue de entrenamiento de cada validacion cruzada y no
filtre informacion del de validacion.

Ademas se le quitan dos columnas: `lago_atitlan` y `estacion_seca`. Junto con
`lago_amatitlan` y `estacion_lluviosa` forman pares que suman exactamente 1 en
cada fila, lo que para un modelo lineal con intercepto es colinealidad perfecta.
Los modelos de arboles no la sufren y conservan las cuatro.

**Random Forest** y **Gradient Boosting** reciben el conjunto completo.

Los tres compensan el desbalance de forma explicita: `class_weight` en los dos
primeros y `scale_pos_weight` en el tercero.

In [4]:
definiciones = md.definir_modelos(datos['y_entrenamiento'])

print(f'Razon negativos / positivos usada en scale_pos_weight: '
      f'{md.peso_clase_positiva(datos["y_entrenamiento"]):.2f}')
print()
for nombre, definicion in definiciones.items():
    columnas = md.columnas_para(nombre, predictores)
    print(f'{nombre}  ->  {len(columnas)} columnas')
    for hiperparametro, valores in sorted(definicion['espacio'].items()):
        print(f'    {hiperparametro}: {valores}')
    print()

Razon negativos / positivos usada en scale_pos_weight: 76.41

regresion_logistica  ->  15 columnas
    modelo__C: [0.01, 0.1, 1.0, 10.0]
    modelo__solver: ['lbfgs', 'liblinear']

random_forest  ->  17 columnas
    max_depth: [None, 12, 20]
    max_features: ['sqrt', 0.5]
    min_samples_leaf: [1, 5, 20]
    n_estimators: [200, 400]

gradient_boosting  ->  17 columnas
    colsample_bytree: [0.8, 1.0]
    learning_rate: [0.03, 0.1, 0.3]
    max_depth: [3, 6, 9]
    n_estimators: [200, 400, 600]
    subsample: [0.8, 1.0]



## 4. Ajuste de hiperparametros

Busqueda aleatoria con validacion cruzada estratificada de 3 pliegues **sobre el
conjunto de entrenamiento**. El conjunto de prueba no participa en ningun momento
de la seleccion: se reserva entero para la comparacion final.

El criterio de seleccion es F2. Elegir por Accuracy premiaría al modelo que
predice siempre ausencia, y elegir por Precision premiaría al que solo se
arriesga con los casos obvios, que es justo lo contrario de lo que interesa
cuando el error costoso es no detectar una floracion.

La celda siguiente reentrena todo desde cero y tarda unos minutos.

In [5]:
import warnings
warnings.filterwarnings('ignore')

salida = md.entrenar_todos()
print()
print(f"Particion:       {Path(salida['particion']).relative_to(ROOT)}")
print(f"Hiperparametros: {Path(salida['hiperparametros']).relative_to(ROOT)}")

Entrenamiento: 344864 observaciones, 4455 positivas (1.2918 %)
Prueba:        147799 observaciones, 1910 positivas (1.2923 %)


- regresion_logistica: F2 de validacion cruzada 0.6956 con {'modelo__solver': 'lbfgs', 'modelo__C': 10.0}


- random_forest: F2 de validacion cruzada 0.9420 con {'n_estimators': 200, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': 20}


- gradient_boosting: F2 de validacion cruzada 0.9550 con {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 9, 'learning_rate': 0.1, 'colsample_bytree': 0.8}

Particion:       data\processed\ml\particion_70_30.parquet
Hiperparametros: results\tables\hiperparametros_modelos.csv


### Que se evaluo y que se eligio

El inciso pide dejar por escrito que hiperparametros se modificaron, que valores
se evaluaron y con que criterio se eligio el modelo final. La tabla siguiente lo
recoge, y la ultima fila de cada modelo es el F2 de validacion cruzada que
justifico la eleccion.

In [6]:
hiperparametros = pd.read_csv(ROOT / 'results' / 'tables' / 'hiperparametros_modelos.csv')
display(hiperparametros.set_index(['modelo', 'hiperparametro']))

valores_evaluados  \
modelo              hiperparametro                                  
regresion_logistica modelo__C              [0.01, 0.1, 1.0, 10.0]   
                    modelo__solver         ["lbfgs", "liblinear"]   
                    f2_validacion_cruzada                     NaN   
random_forest       max_depth                      [null, 12, 20]   
                    max_features                    ["sqrt", 0.5]   
                    min_samples_leaf                   [1, 5, 20]   
                    n_estimators                       [200, 400]   
                    f2_validacion_cruzada                     NaN   
gradient_boosting   colsample_bytree                   [0.8, 1.0]   
                    learning_rate                [0.03, 0.1, 0.3]   
                    max_depth                           [3, 6, 9]   
                    n_estimators                  [200, 400, 600]   
                    subsample                          [0.8, 1.0]   
                    f2_validacion_cruzada                     NaN   

                                          valor_elegido  
modelo              hiperparametro                       
regresion_logistica modelo__C                      10.0  
                    modelo__solver              "lbfgs"  
                    f2_validacion_cruzada      0.695609  
random_forest       max_depth                        20  
                    max_features                 "sqrt"  
                    min_samples_leaf                  5  
                    n_estimators                    200  
                    f2_validacion_cruzada      0.941993  
gradient_boosting   colsample_bytree                0.8  
                    learning_rate                   0.1  
                    max_depth                         9  
                    n_estimators                    200  
                    subsample                       0.8  
                    f2_validacion_cruzada      0.955028

## 5. Verificacion del contrato

Comprueba que la particion persistida se reproduce con la semilla declarada, que
es estratificada, que entrenamiento y prueba no se solapan y cubren toda la
matriz, que `lago` y `fecha` coinciden con el dataset base, y que los tres
modelos existen y se pueden cargar.

Este es el comando que deben ejecutar los ejercicios posteriores antes de
reutilizar esta particion.

In [7]:
verificacion = md.verificar_modelos()
print(f"Entrenamiento:      {verificacion['entrenamiento']:,}")
print(f"Prueba:             {verificacion['prueba']:,}")
print(f"Positivas en prueba:{verificacion['positivos_prueba']:,}")
print(f"Modelos:            {', '.join(verificacion['modelos'])}")

Entrenamiento:      344,864
Prueba:             147,799
Positivas en prueba:1,910
Modelos:            regresion_logistica, random_forest, gradient_boosting
